In [98]:
# NAAnoBot training prototype
# Note: This training script is essentially identical to what I used on the GPU cloud, just with more lightweight parameters
# Mainly used this to monitor loss descent (or oscillation) after each model architecture edit / feature engineering iteration. Sometimes halted training and saved partially-trained model for prototyping

In [1]:
import torch
import pandas as pd
%matplotlib inline

from src.paths import *
from src.nano_maker.modules.data_handling.naano_dataset import NAAnoDataset
from src.nano_maker.naanobot import NAAnoBot

In [2]:
# Load data
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'
PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "database"

RADIAL_SEQUENCES = pd.read_pickle(DATAPATH / "radial_seq_df.pkl")
RADIAL_SEQUENCES.set_index("PDB_ID", inplace=True)

MOLECULAR_FINGERPRINTS = pd.read_pickle(DATAPATH / "molfp_df.pkl")
MOLECULAR_FINGERPRINTS.set_index("smiles_str", inplace=True)

train_pointers_path = DATAPATH / "naanobot_training_pointers_dict.parquet"
test_pointers_path = DATAPATH / "naanobot_test_pointers_dict.parquet"

In [21]:
n_spatial_features = 10

radial_resolution = 100
max_angstroms = 33

block_size = 40     # make this 75
batch_size = 64    # make ths 128
n_embd = 256        # 512
n_head = 4          # 8
n_layers = 2        # 6
map4_res = 1024
dropout=0.15
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_epochs = 1        # 3
learning_rate = 1e-4
loss_temperature = 0.2


# training dataset
training_dataset = NAAnoDataset(pointer_path=train_pointers_path,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms,
                                )
# DataLoader
from torch.utils.data import DataLoader
loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

torch.manual_seed(311104)

# test / validation set
test_set = NAAnoDataset(pointer_path=test_pointers_path,
                                 smiles_molfp=MOLECULAR_FINGERPRINTS,
                                 pdb_radial=RADIAL_SEQUENCES,
                                 block_size=block_size,
                                 max_angstroms=max_angstroms,
                        )# DataLoader
from torch.utils.data import DataLoader
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True
)

In [29]:
model = NAAnoBot(n_embd=n_embd, n_head=n_head, n_layers=n_layers,
                 block_size=block_size, map4_res=map4_res, max_angstroms=max_angstroms, n_spatial_features=n_spatial_features,
                 loss_temperature=loss_temperature,
                 dropout=dropout).to(device)

In [30]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
model.train()

NAAnoBot(
  (nAAno_project): Linear(in_features=31, out_features=256, bias=True)
  (pos_emb): Embedding(40, 256)
  (mol_encoder): MolecularEncoder(
    (block1): Sequential(
      (0): Linear(in_features=1024, out_features=1024, bias=True)
      (1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.15, inplace=False)
    )
    (check1): Linear(in_features=1024, out_features=512, bias=True)
    (block2): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Dropout(p=0.15, inplace=False)
    )
    (check2): Linear(in_features=512, out_features=256, bias=True)
    (residual1): Linear(in_features=1024, out_features=512, bias=True)
    (block3): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2):

In [31]:
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=1000
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(loader) * n_epochs,
    eta_min=3e-5
)

lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[1000]
)

In [32]:
@torch.no_grad()
def estimate_loss(model, loader, device, radial_res, max_batches=None):
    model.eval()
    total_loss = 0
    n_batches = 0

    for batch_idx, batch in enumerate(loader):
        if max_batches and batch_idx >= max_batches:
            break

        map4_fp, naano_X, naano_Y, naano_Y_idx = batch

        map4_fp = map4_fp.to(device)
        naano_X = naano_X.to(device)
        naano_Y = naano_Y.to(device)
        naano_Y_idx = naano_Y_idx.to(device)

        _, loss = model(naano_X, map4_fp, naano_Y, naano_Y_idx)
        total_loss += loss.item()
        n_batches += 1

    model.train()
    return total_loss / n_batches

In [33]:
loss_record = []

for epoch in range(n_epochs):   # start with few epochs
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        map4_fp, naano_X, naano_Y, naano_Y_idx = batch

        map4_fp = map4_fp.to(device)
        naano_X = naano_X.to(device)
        naano_Y = naano_Y.to(device)
        naano_Y_idx = naano_Y_idx.to(device)


        optimizer.zero_grad()
        pred, loss = model(naano_X, map4_fp, naano_Y, naano_Y_idx)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()

        loss_record.append(loss.item())
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch + 1} | Batch {batch_idx + 1} | Loss: {loss.item():.6f}")

    train_loss = total_loss / len(loader)
    val_loss = estimate_loss(model, test_loader, device, radial_res=radial_resolution, max_batches=100)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'loss_record': loss_record,
    }, DATAPATH / f"naanobot_prototype_epoch_{epoch+1}.pt")

    print(f"Checkpoint saved ? checkpoints/epoch_{epoch+1}.pt")
    print(f"Epoch {epoch+1} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Gap: {val_loss - train_loss:.6f}")

Epoch 1 | Batch 1 | Loss: 34.303486
Epoch 1 | Batch 51 | Loss: 26.565348
Epoch 1 | Batch 101 | Loss: 18.300461


KeyboardInterrupt: 

In [ ]:
# can halt training and save the partially-trained prototype here

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': lr_scheduler.state_dict(),
}, CONTAINER / 'naanobot_prototypeI4.pt')

In [ ]:
# you can then set the config to look for the prototype.pt file you just made and create your own pockets.